```python
# Copyright (c) 2026 Louis Gevers
# SPDX-License-Identifier: BSD-3-Clause
```

# Legged Robots Project - Task 2: Flat Terrain Analysis

In this notebook you will analyse the data produced by `01_flat_terrain.ipynb`:
training logs and evaluation rollouts.
There are no simulations here, so it does not need a GPU and can be restarted at any time.

**Before you start:** check out the metrics functions in `src/lr_project/metrics.py`.
This is the only source file that you will have to edit.
We provided two example implementations, but it is up to you to implement the relevant metrics using rollout data (see 2.0c) for your analyses.
There are two kinds of functions, recognizable by their output shape (`N` number of robots, `T` number of timesteps):
- **per-step signals**, shape `(N, T)`: one value per robot and per time step, e.g. to plot a quantity over time.
- **per-robot metrics**, shape `(N,)`: one value per robot, summarizing its whole rollout.

Thanks to `%autoreload`, edits to `metrics.py` are picked up without restarting the kernel.

**Rules for your analysis**
- Every number and figure must come from a cell in this notebook. 
Never copy numbers by hand.
- The sections below list the **minimum** we expect. 
Add any analysis, figure or metric that helps you answer the questions.
- Label every figure: axes with units, a legend when there are several curves, and a title.
- Write your answers under the **[ANSWER]** cells in markdown.
Remember that short and precise beats long.

In [ ]:
%load_ext autoreload
%autoreload 2

import math
import numpy as np
import matplotlib.pyplot as plt

from lr_project.rollout import Rollout
from lr_project import registry, analysis, metrics

## 2.0 Collected data

We provide you with an example of how to load three kinds of data you have generated:
the list of runs, the training logs of a run, and its evaluation rollouts.
Run it as is, then reuse the patterns in your analysis sections.

### 2.0a Selecting runs of interest

`registry.runs` lists runs filtered by experiment and variant (the names used in `train.train`), sorted by creation time. 
`runs[-1]` is the latest run (it's a python list).

In [ ]:
runs = registry.runs(
    experiment="flat",
    variant="baseline",
)
registry.show(runs)

In [ ]:
run = runs[-1]
registry.show([run])

### 2.0b Training data

`analysis.load_training_logs` reads the curves logged during training and returns a dictionary `{curve name: (iterations, values)}`. 
`patterns` are regular expressions over the curve names, e.g. `"Episode_Reward/.*"` for all reward terms.
The curve names match the ones you see in TensorBoard.

In [ ]:
train_metrics = analysis.load_training_logs(
    run_dir=run.path,
    patterns=["Train/mean_reward", "Train/mean_episode_length", "Episode_Reward/.*"],
)

print("Loaded training metrics:")
for metric in train_metrics.keys():
    print(f"- {metric!r}")

Below we provide an example of plotting training logs, specifically the mean reward per learning iteration:

In [ ]:
iterations, mean_reward = train_metrics["Train/mean_reward"]
print(f"Loaded {len(iterations)} iterations of mean reward with last 3 elements: {mean_reward[-3:]}")

fig, ax = plt.subplots()

ax.plot(iterations, mean_reward)
ax.set_xlabel("Learning iteration")
ax.set_ylabel("Mean reward")

fig.tight_layout()
plt.show()

### 2.0c Evaluation data

`Rollout.load` reads an evaluation saved in notebook 01. 
A `Rollout` holds numpy arrays for N robots and T time steps (see `src/lr_project/rollout.py` for all fields), for example:

| Field | Shape | Content |
| --- | --- | --- |
| `cmd` | (N, 3) | command of each robot $(v_x, v_y, \omega_z)$ |
| `alive` | (N,) | `True` if the robot never fell |
| `base_lin_vel`, `base_ang_vel` | (N, T, 3) | base velocities in the body frame |
| `base_pos`, `base_quat` | (N, T, 3), (N, T, 4) | base position and orientation in the world frame |
| `joint_pos`, `joint_vel`, `joint_torque` | (N, T, 12) | joint states and torques |
| `foot_contact`, `foot_height` | (N, T, 4) | contact flags and heights of the feet (FL, FR, RL, RR) |
| `foot_grf` | (N, T, 4, 3) | ground reaction forces on the feet |

Robots that fell distort most metrics: compute them on `rollouts.select(rollouts.alive)` and report the survival rate separately.

Below we provide some examples of how to take statistics from rollout data and making a simple figure:

In [ ]:
rollouts = Rollout.load(run.eval_file("command_grid"))

n_robot = len(rollouts.alive)
survive_count = np.sum(rollouts.alive)
survival_rate = np.mean(rollouts.alive)

lin_vel_error = metrics.lin_vel_error(rollouts)

print(f"Survival rate is {survival_rate:.1%} ({np.sum(rollouts.alive)} out of {n_robot} evaluations alive)")
print(f"Average linear velocity error is {np.mean(lin_vel_error):.2f} (std={np.std(lin_vel_error):.2f})")

robot_id = 0
print(f"Robot {robot_id} command: {rollouts.cmd[robot_id]} (lin vel error: {lin_vel_error[robot_id]:.2f})")

fig, axes = plt.subplots(nrows=1, ncols=2)

time = rollouts.dt * np.arange(0, rollouts.base_pos.shape[1])
base_x = rollouts.base_pos[robot_id, :, 0]
base_y = rollouts.base_pos[robot_id, :, 1]
base_z = rollouts.base_pos[robot_id, :, 2]

axes[0].plot(base_x, base_y)
axes[0].set_aspect("equal")
axes[0].set_xlabel("x (m)")
axes[0].set_ylabel("y (m)")

axes[1].plot(time, base_z)
axes[1].set_xlabel("time (s)")
axes[1].set_ylabel("z (m)")

fig.suptitle(f"Robot {robot_id} base trajectory")
fig.tight_layout()
plt.show()

## 2.1 Baseline performance

In this section you will characterize your baseline policy: how it learned, how well it performs, and discuss whether it could run on a real robot.

First load the training and rollout data:

In [ ]:
# Helpers shared by sections 2.1 and 2.2 (notebook 02 does not import notebook 01)
import warnings
import yaml
from IPython.display import Markdown, display

SEEDS = [42, 1, 2]  # training seeds of every configuration (notebook 01)
EVAL_NAME = "command_grid"  # same evaluation for every run (notebook 01, 1.4b and 1.5f)
FEET = ("FL", "FR", "RL", "RR")
LAST = 20  # "final" value of a training curve = mean of the last 20 iterations
TRAIN_PATTERNS = [
    "Train/mean_reward",
    "Train/mean_episode_length",
    "Episode_Reward/.*",
    "Episode_Termination/.*",
    "Metrics/.*error_vel_.*",
]
# Effort limits of the Go2 actuators (go2_constants.py), joint order of go2.xml:
# FL, FR, RL, RR legs x (hip, thigh, calf)
EFFORT_LIMIT = np.tile([23.5, 23.5, 45.0], 4)  # N m

# Colors: one per configuration, one per foot (fixed order)
COLORS = {"baseline": "#2a78d6", "no_base_lin_vel": "#eb6834"}
FOOT_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]


def run_seed(r) -> int:
    """Seed of a run, read from the config saved with it."""
    return yaml.unsafe_load((r.path / "params" / "agent.yaml").read_text())["seed"]


def runs_per_seed(variant: str) -> dict:
    """Latest complete run of `variant` for each seed in SEEDS (seed -> Run).
    Runs are sorted by creation time, so a rerun replaces an older run of the same seed;
    interrupted runs (no model_499.pt) are ignored."""
    found = {}
    for r in registry.runs(experiment="flat", variant=variant):
        if any(r.path.glob("model_499.pt")) and run_seed(r) in SEEDS:
            found[run_seed(r)] = r
    return found


def load_config(variant: str) -> dict:
    """seed -> {"run", "logs" (training curves), "all" (rollout), "alive" (robots that never fell)}."""
    runs = runs_per_seed(variant)
    missing = [s for s in SEEDS if s not in runs]
    if missing:
        raise FileNotFoundError(f"{variant}: no complete run for seeds {missing}")
    data = {}
    for seed in SEEDS:
        r = runs[seed]
        rollout = Rollout.load(r.eval_file(EVAL_NAME))
        data[seed] = {
            "run": r,
            "logs": analysis.load_training_logs(r.path, TRAIN_PATTERNS),
            "all": rollout,
            "alive": rollout.select(rollout.alive),
        }
    return data


def nanmean(x, axis=None):
    """np.nanmean without the warning for all-NaN slices (e.g. CoT of standing robots)."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        return np.nanmean(x, axis=axis)


def nanstd(x, axis=None):
    """Sample std over seeds (ddof=1), ignoring NaN, without warnings."""
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        return np.nanstd(x, axis=axis, ddof=1)


# ---------- per-robot metrics, shape (N,) ----------
def mean_power(r):
    """Mean mechanical power, in W."""
    return np.mean(metrics.mechanical_power(r), axis=1)


def torque_ratio_p99(r):
    """99th percentile over time and joints of |torque| / effort limit (1 = saturation)."""
    return np.percentile(np.abs(r.joint_torque) / EFFORT_LIMIT, 99, axis=(1, 2))


def joint_speed_p99(r):
    """99th percentile over time and joints of |joint velocity|, in rad/s."""
    return np.percentile(np.abs(r.joint_vel), 99, axis=(1, 2))


METRICS = {  # name: (per-robot function, unit)
    "lin_vel_error": (metrics.lin_vel_error, "m/s"),
    "yaw_rate_error": (metrics.yaw_rate_error, "rad/s"),
    "cost_of_transport": (metrics.cost_of_transport, "-"),
    "mean_power": (mean_power, "W"),
    "base_ang_vel_xy": (metrics.base_ang_vel_xy, "rad/s"),
    "base_tilt": (metrics.base_tilt, "deg"),
    "base_height_std": (metrics.base_height_std, "m"),
    "joint_acceleration": (metrics.joint_acceleration, "rad/s^2"),
    "action_rate": (metrics.action_rate, "-"),
    "duty_factor": (metrics.duty_factor, "-"),
    "stride_frequency": (metrics.stride_frequency, "Hz"),
    "stride_length": (metrics.stride_length, "m"),
    "foot_clearance": (metrics.foot_clearance, "m"),
    "peak_vertical_grf": (metrics.peak_vertical_grf, "body weights"),
    "weight_support": (metrics.weight_support, "body weights"),
    "torque_ratio_p99": (torque_ratio_p99, "-"),
    "joint_speed_p99": (joint_speed_p99, "rad/s"),
}


def robot_metrics(r) -> dict:
    """name -> per-robot values, shape (N,)."""
    return {name: fn(r) for name, (fn, _) in METRICS.items()}


# ---------- commands ----------
CATEGORIES = ["stand", "turn in place", "linear", "linear + turn"]


def command_category(c) -> str:
    moving = c[0] != 0 or c[1] != 0
    if not moving:
        return "stand" if c[2] == 0 else "turn in place"
    return "linear" if c[2] == 0 else "linear + turn"


def command_label(c) -> str:
    return f"({c[0]:g}, {c[1]:g}, {c[2]:g})"


def per_command(values, r, commands):
    """Mean of a per-robot metric for each command (NaN if no robot of that command survived)."""
    out = []
    for c in commands:
        mask = np.all(np.isclose(r.cmd, c), axis=1)
        out.append(nanmean(values[mask]) if mask.any() else np.nan)
    return np.array(out)


def per_category(values, r):
    cats = np.array([command_category(c) for c in r.cmd])
    return np.array([nanmean(values[cats == k]) if (cats == k).any() else np.nan for k in CATEGORIES])


def moving_mask(r):
    return np.linalg.norm(r.cmd[:, :2], axis=1) > 0


# ---------- statistics over seeds ----------
def seed_means(per_seed_metrics: dict, name: str) -> np.ndarray:
    """Mean over robots of one metric, for each seed. Shape (len(SEEDS),)."""
    return np.array([nanmean(per_seed_metrics[s][name]) for s in SEEDS])


def fmt(values, digits=3) -> str:
    """mean ± sample std over seeds."""
    values = np.asarray(values, dtype=float)
    return f"{np.mean(values):.{digits}g} ± {np.std(values, ddof=1):.{digits}g}"


def show_table(header, rows):
    lines = ["| " + " | ".join(header) + " |", "|" + " --- |" * len(header)]
    lines += ["| " + " | ".join(str(x) for x in row) + " |" for row in rows]
    display(Markdown("\n".join(lines)))


# ---------- training curves ----------
def find_tag(data, regex) -> str:
    import re

    tags = data[SEEDS[0]]["logs"].keys()
    return next(t for t in tags if re.fullmatch(regex, t))


def curves(data, tag):
    """Iterations (n,) and values (seeds, n) of one training curve, cut to the shortest run."""
    n = min(len(data[s]["logs"][tag][0]) for s in SEEDS)
    it = data[SEEDS[0]]["logs"][tag][0][:n]
    return it, np.stack([data[s]["logs"][tag][1][:n] for s in SEEDS])


def final_values(data, tag) -> np.ndarray:
    """Mean of the last LAST iterations, for each seed."""
    return np.array([np.mean(data[s]["logs"][tag][1][-LAST:]) for s in SEEDS])


def plot_band(ax, data, tag, label=None, color=None):
    """Mean ± std over seeds of a training curve."""
    it, v = curves(data, tag)
    m, sd = v.mean(0), v.std(0, ddof=1)
    ax.plot(it, m, color=color, lw=1.5, label=label)
    ax.fill_between(it, m - sd, m + sd, color=color, alpha=0.25, lw=0)


def progress_iteration(it, values, frac, window=10):
    """First iteration at which the smoothed curve has covered `frac` of its change
    from the start to its final value (mean of the last LAST iterations)."""
    smooth = np.convolve(values, np.ones(window) / window, mode="valid")
    start, end = smooth[0], np.mean(values[-LAST:])
    if np.isclose(end, start):
        return np.nan
    reached = np.flatnonzero((smooth - start) / (end - start) >= frac)
    return it[window - 1 :][reached[0]] if len(reached) else np.nan

In [ ]:
baseline = load_config("baseline")

for seed, d in baseline.items():
    r = d["all"]
    print(
        f"seed {seed}: run {d['run'].path.name}, {r.num_robots} robots x {r.base_lin_vel.shape[1]} steps "
        f"(dt = {r.dt} s), survival {np.mean(r.alive):.1%} ({r.alive.sum()}/{r.num_robots})"
    )
    fell = r.cmd[~r.alive]
    if len(fell):
        labels, counts = np.unique([command_label(c) for c in fell], return_counts=True)
        print("   fallen robots per command:", dict(zip(labels, counts.tolist())))

# Every seed must have been evaluated with the same settings (only the checkpoint differs)
EVAL_META = {k: v for k, v in baseline[SEEDS[0]]["all"].meta.items() if k != "checkpoint"}
for seed, d in baseline.items():
    assert {k: v for k, v in d["all"].meta.items() if k != "checkpoint"} == EVAL_META, seed
COMMANDS = [tuple(c) for c in EVAL_META["commands"]]
print(f"\nEvaluation: {len(COMMANDS)} commands x {EVAL_META['robots_per_command']} robots, "
      f"{EVAL_META['duration_s']} s, evaluation seed {EVAL_META['seed']}")

# Metrics are computed on the robots that never fell
baseline_metrics = {s: robot_metrics(d["alive"]) for s, d in baseline.items()}
baseline_survival = np.array([np.mean(baseline[s]["all"].alive) for s in SEEDS])
print(f"Survival rate over seeds: {fmt(100 * baseline_survival)} %")

### 2.1a Training performance

**Minimum:** mean reward and mean episode length curves, and the curves of the individual reward terms.

Questions: 
- When does the policy converge, and would more training help? 
- Which reward terms dominate, and which improve first? 
- Explain how you arrived at your baseline: which settings you tuned, what failures you observed along the way, and how you fixed them.

In [ ]:
# Figure 1: mean reward and episode length, mean ± std over the 3 seeds
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
plot_band(axes[0], baseline, "Train/mean_reward", label="baseline (3 seeds)", color=COLORS["baseline"])
axes[0].set(xlabel="Learning iteration", ylabel="Mean episode reward (-)", title="Mean reward")
plot_band(axes[1], baseline, "Train/mean_episode_length", label="baseline (3 seeds)", color=COLORS["baseline"])
axes[1].axhline(1000, color="gray", ls="--", lw=1, label="time-out (20 s, no fall)")
axes[1].set(xlabel="Learning iteration", ylabel="Mean episode length (steps of 0.02 s)",
            title="Mean episode length")
for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend(loc="lower right")
fig.suptitle("Baseline training (line: mean over seeds 42, 1, 2; band: ± 1 std)")
fig.tight_layout()
plt.show()

# Figure 2: every reward term
terms = sorted(t for t in baseline[SEEDS[0]]["logs"] if t.startswith("Episode_Reward/"))
ncols = 4
nrows = math.ceil(len(terms) / ncols)
fig, axes = plt.subplots(nrows, ncols, figsize=(14, 2.8 * nrows), squeeze=False)
for ax, tag in zip(axes.flat, terms):
    plot_band(ax, baseline, tag, label="baseline (3 seeds)", color=COLORS["baseline"])
    ax.axhline(0, color="gray", lw=0.8)
    ax.set_title(tag.removeprefix("Episode_Reward/"), fontsize=10)
    ax.set_xlabel("Learning iteration")
    ax.set_ylabel("Reward per episode (-)")
    ax.grid(alpha=0.3)
for ax in axes.flat[len(terms):]:
    ax.axis("off")
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="lower right", fontsize=9)
fig.suptitle("Baseline reward terms (weighted, summed over the episode; mean ± std over seeds)")
fig.tight_layout()
plt.show()

In [ ]:
# Convergence: iteration at which the smoothed mean reward covers 95 % of its total increase
it, rew = curves(baseline, "Train/mean_reward")
conv = np.array([progress_iteration(it, v, 0.95) for v in rew])
print(f"Mean reward reaches 95 % of its final value at iteration {fmt(conv)} (seeds {SEEDS}: {conv.tolist()})")
final_rew = final_values(baseline, "Train/mean_reward")
print(f"Final mean reward (last {LAST} iterations): {fmt(final_rew)}")

# Would more training help? Linear trend of the mean reward over the last 100 iterations
slopes = np.array([np.polyfit(it[-100:], v[-100:], 1)[0] * 100 for v in rew])
print(f"Trend over the last 100 iterations: {fmt(slopes)} reward per 100 iterations "
      f"({fmt(100 * slopes / final_rew)} % of the final reward)")

ep_len = final_values(baseline, "Train/mean_episode_length")
print(f"Final mean episode length: {fmt(ep_len)} steps (1000 = no fall in 20 s)")
for tag in ("Metrics/.*error_vel_xy", "Metrics/.*error_vel_yaw"):
    t = find_tag(baseline, tag)
    print(f"Final {t}: {fmt(final_values(baseline, t))}")

# Reward terms: final value, share of the total magnitude, and how early each term converged
finals = {t: final_values(baseline, t) for t in terms}
total = sum(np.abs(v.mean()) for v in finals.values())
rows = []
for t in sorted(terms, key=lambda t: -abs(finals[t].mean())):
    it_t, v_t = curves(baseline, t)
    p90 = np.array([progress_iteration(it_t, v, 0.9) for v in v_t])
    rows.append([t.removeprefix("Episode_Reward/"), fmt(finals[t]),
                 f"{100 * abs(finals[t].mean()) / total:.1f} %", f"{np.nanmean(p90):.0f}"])
show_table(["reward term", f"final value (last {LAST} it.)", "share of total |reward|",
            "iteration at 90 % of its change"], rows)

**[ANSWER]** *Write your answer to the questions here, 2 paragraphs max*

### 2.1b Evaluation performance

**Minimum:**
- velocity tracking: linear and yaw rate errors, overall and per command
- cost of transport
- base stability, with a metric of your own design
- foot ground reaction forces
- a foot contact plot (which foot is on the ground over time)

Questions: 
- Which commands are tracked well, and which are hardest? 
- Which gait did the policy learn, and how can you tell from the contact plot? 
- Is the motion efficient and smooth?

In [ ]:
# Overall evaluation metrics (robots that never fell): mean over robots per seed, then mean ± std over seeds
rows = [["survival rate", "%", fmt(100 * baseline_survival)] + [f"{100 * v:.1f}" for v in baseline_survival]]
for name, (_, unit) in METRICS.items():
    v = seed_means(baseline_metrics, name)
    rows.append([name, unit, fmt(v)] + [f"{x:.3g}" for x in v])
show_table(["metric", "unit", "mean ± std over seeds"] + [f"seed {s}" for s in SEEDS], rows)
print("cost_of_transport and stride_length are averaged over moving robots only (NaN below 0.05 m/s).")

In [ ]:
# Figure 3: tracking errors per command (mean over the 3 seeds)
def command_matrix(per_seed_metrics, data, name):
    """(len(SEEDS), len(COMMANDS)) per-command mean of a metric."""
    return np.stack([per_command(per_seed_metrics[s][name], data[s]["alive"], COMMANDS) for s in SEEDS])


lin_cmd = command_matrix(baseline_metrics, baseline, "lin_vel_error")
yaw_cmd = command_matrix(baseline_metrics, baseline, "yaw_rate_error")
vxs = sorted({c[0] for c in COMMANDS})
vys = sorted({c[1] for c in COMMANDS})
wzs = sorted({c[2] for c in COMMANDS})

fig, axes = plt.subplots(2, len(wzs), figsize=(12, 7), constrained_layout=True)
for row, (values, name, unit, cmap) in enumerate([
    (lin_cmd.mean(0), "Linear velocity error", "m/s", "Blues"),
    (yaw_cmd.mean(0), "Yaw rate error", "rad/s", "Oranges"),
]):
    vmax = np.nanmax(values)
    for col, wz in enumerate(wzs):
        grid = np.full((len(vys), len(vxs)), np.nan)
        for k, c in enumerate(COMMANDS):
            if c[2] == wz:
                grid[vys.index(c[1]), vxs.index(c[0])] = values[k]
        ax = axes[row, col]
        im = ax.imshow(grid, origin="lower", cmap=cmap, vmin=0, vmax=vmax)
        for i in range(len(vys)):
            for j in range(len(vxs)):
                ax.text(j, i, f"{grid[i, j]:.2f}", ha="center", va="center",
                        color="white" if grid[i, j] > 0.6 * vmax else "black", fontsize=10)
        ax.set_xticks(range(len(vxs)), [f"{v:g}" for v in vxs])
        ax.set_yticks(range(len(vys)), [f"{v:g}" for v in vys])
        ax.set(xlabel="commanded vx (m/s)", ylabel="commanded vy (m/s)",
               title=f"{name}, wz = {wz:g} rad/s")
    fig.colorbar(im, ax=axes[row], label=f"RMS error ({unit})", shrink=0.9)
fig.suptitle("Baseline tracking error per command (RMS over 10 s, mean over robots and seeds)")
plt.show()

# Hardest commands
for values, name, unit in [(lin_cmd, "linear velocity error", "m/s"), (yaw_cmd, "yaw rate error", "rad/s")]:
    order = np.argsort(-values.mean(0))
    print(f"Highest {name} (mean ± std over seeds, {unit}):")
    for k in order[:5]:
        print(f"   {command_label(COMMANDS[k]):>16} [{command_category(COMMANDS[k])}]: {fmt(values[:, k])}")
    k = order[-1]
    print(f"   lowest: {command_label(COMMANDS[k])}: {fmt(values[:, k])}")

In [ ]:
# Figure 4: tracking, efficiency and base stability per command category
def category_matrix(per_seed_metrics, data, name):
    """(len(SEEDS), len(CATEGORIES)) per-category mean of a metric."""
    return np.stack([per_category(per_seed_metrics[s][name], data[s]["alive"]) for s in SEEDS])


PANELS = ["lin_vel_error", "yaw_rate_error", "cost_of_transport",
          "base_ang_vel_xy", "base_tilt", "joint_acceleration"]
x = np.arange(len(CATEGORIES))
fig, axes = plt.subplots(2, 3, figsize=(13, 6.5))
for ax, name in zip(axes.flat, PANELS):
    v = category_matrix(baseline_metrics, baseline, name)
    ax.bar(x, nanmean(v, axis=0), yerr=nanstd(v, axis=0), width=0.6,
           color=COLORS["baseline"], capsize=4, label="baseline (mean ± std over seeds)")
    for s in range(len(SEEDS)):
        ax.plot(x, v[s], "o", color="black", ms=3, label="single seed" if s == 0 else None)
    ax.set_xticks(x, CATEGORIES, rotation=15)
    ax.set_ylabel(f"{name} ({METRICS[name][1]})")
    ax.set_title(name)
    ax.grid(axis="y", alpha=0.3)
axes.flat[0].legend(fontsize=8)
fig.suptitle("Baseline metrics per command category (CoT undefined when not translating)")
fig.tight_layout()
plt.show()

rows = []
for name in PANELS:
    v = category_matrix(baseline_metrics, baseline, name)
    rows.append([name, METRICS[name][1]] + [fmt(v[:, k]) if np.isfinite(v[:, k]).all() else "-"
                                            for k in range(len(CATEGORIES))])
show_table(["metric", "unit"] + CATEGORIES, rows)

In [ ]:
# Example robots (seed 42): forward walking and turning in place, steady-state window
WINDOW = (5.0, 7.0)  # s, after the start-up transient


def example_robot(data, seed, command):
    r = data[seed]["alive"]
    idx = np.flatnonzero(np.all(np.isclose(r.cmd, command), axis=1))
    if not len(idx):
        raise ValueError(f"no surviving robot with command {command} for seed {seed}")
    return r, idx[0]


r, i = example_robot(baseline, 42, (1.0, 0.0, 0.0))
t = r.dt * np.arange(r.base_lin_vel.shape[1])
sel = (t >= WINDOW[0]) & (t <= WINDOW[1])
weight = metrics.body_weight(r)[i]

# Figure 5: vertical ground reaction forces
fig, axes = plt.subplots(1, 2, figsize=(13, 3.8), gridspec_kw={"width_ratios": [2, 1]})
for f, foot in enumerate(FEET):
    axes[0].plot(t[sel], np.abs(r.foot_grf[i, sel, f, 2]) / weight, color=FOOT_COLORS[f], lw=1.3, label=foot)
axes[0].plot(t[sel], metrics.total_vertical_grf(r)[i, sel], color="black", ls="--", lw=1, label="sum of 4 feet")
axes[0].axhline(1, color="gray", lw=0.8)
axes[0].set(xlabel="time since rollout start (s)", ylabel="vertical GRF (body weights)",
            title="Vertical GRF per foot, command (1, 0, 0), seed 42")
axes[0].legend(ncol=5, fontsize=8, loc="upper right")
axes[0].grid(alpha=0.3)

x = np.arange(len(CATEGORIES))
for k, (name, offset) in enumerate([("peak_vertical_grf", -0.2), ("weight_support", 0.2)]):
    v = category_matrix(baseline_metrics, baseline, name)
    axes[1].bar(x + offset, nanmean(v, axis=0), yerr=nanstd(v, axis=0), width=0.38,
                color=["#2a78d6", "#1baf7a"][k], capsize=3, label=name)
axes[1].axhline(1, color="gray", lw=0.8)
axes[1].set_xticks(x, CATEGORIES, rotation=15)
axes[1].set(ylabel="body weights", title="GRF per command category")
axes[1].legend(fontsize=8)
axes[1].grid(axis="y", alpha=0.3)
fig.tight_layout()
plt.show()

ws = seed_means(baseline_metrics, "weight_support")
pk = seed_means(baseline_metrics, "peak_vertical_grf")
print(f"Weight support (sanity check, should be about 1): {fmt(ws)} body weights")
print(f"Peak vertical GRF per stance phase: {fmt(pk)} body weights")

In [ ]:
# Figure 6: foot contact plot (gait diagram)
def contact_segments(contact, dt):
    """(start time, duration) of every contact phase of a 1-D contact signal."""
    c = np.concatenate([[0], contact.astype(int), [0]])
    d = np.diff(c)
    starts, ends = np.flatnonzero(d == 1), np.flatnonzero(d == -1)
    return [(s * dt, (e - s) * dt) for s, e in zip(starts, ends)]


def plot_contacts(ax, r, i, window, title):
    t0 = int(round(window[0] / r.dt))
    t1 = int(round(window[1] / r.dt))
    for f, foot in enumerate(FEET):
        segs = [(window[0] + s, w) for s, w in contact_segments(r.foot_contact[i, t0:t1, f], r.dt)]
        ax.broken_barh(segs, (f - 0.35, 0.7), color=FOOT_COLORS[f])
    ax.set_yticks(range(len(FEET)), FEET)
    ax.invert_yaxis()
    ax.set(xlim=window, xlabel="time since rollout start (s)", ylabel="foot (bar = in contact)", title=title)
    ax.grid(axis="x", alpha=0.3)


fig, axes = plt.subplots(2, 1, figsize=(11, 5), sharex=True)
for ax, cmd in zip(axes, [(1.0, 0.0, 0.0), (0.0, 0.0, 1.0)]):
    r, i = example_robot(baseline, 42, cmd)
    plot_contacts(ax, r, i, WINDOW, f"Baseline seed 42, command {command_label(cmd)}")
axes[0].set_xlabel("")
fig.suptitle("Foot contacts over time (FL front left, FR front right, RL rear left, RR rear right)")
fig.tight_layout()
plt.show()

# Gait identification: fraction of time two feet are in the same contact state (moving commands)
PAIRS = {"diagonal FL-RR": (0, 3), "diagonal FR-RL": (1, 2), "front FL-FR": (0, 1),
         "hind RL-RR": (2, 3), "left FL-RL": (0, 2), "right FR-RR": (1, 3)}


def pair_sync(data):
    """pair -> (seeds,) fraction of time the two feet are both in contact or both in swing."""
    out = {}
    for name, (a, b) in PAIRS.items():
        out[name] = np.array([
            np.mean(d["alive"].foot_contact[moving_mask(d["alive"])][..., a]
                    == d["alive"].foot_contact[moving_mask(d["alive"])][..., b])
            for d in (data[s] for s in SEEDS)
        ])
    return out


baseline_sync = pair_sync(baseline)
show_table(["foot pair", "same contact state (fraction of time, mean ± std over seeds)"],
           [[k, fmt(v)] for k, v in baseline_sync.items()])
print("Trot: diagonal pairs about 1, other pairs low. Pace: left/right pairs high. Bound: front/hind pairs high.")
print(f"Duty factor: {fmt(seed_means(baseline_metrics, 'duty_factor'))}, "
      f"stride frequency: {fmt(seed_means(baseline_metrics, 'stride_frequency'))} Hz, "
      f"foot clearance: {fmt(seed_means(baseline_metrics, 'foot_clearance'))} m")

In [ ]:
# Motor demands (for 2.1c): torques against the actuator limits, joint speeds, smoothness
all_torque = np.concatenate([np.abs(baseline[s]["alive"].joint_torque).reshape(-1, 12) for s in SEEDS])
all_speed = np.concatenate([np.abs(baseline[s]["alive"].joint_vel).reshape(-1, 12) for s in SEEDS])
rows = []
for j, joint in enumerate(["hip", "thigh", "calf"]):
    cols = [j, j + 3, j + 6, j + 9]
    tq, sp = all_torque[:, cols], all_speed[:, cols]
    rows.append([joint, f"{EFFORT_LIMIT[j]:g}", f"{np.percentile(tq, 99):.1f}", f"{tq.max():.1f}",
                 f"{100 * np.mean(tq > 0.95 * EFFORT_LIMIT[j]):.2f} %",
                 f"{np.percentile(sp, 99):.1f}", f"{sp.max():.1f}"])
show_table(["joint", "effort limit (N m)", "|torque| p99 (N m)", "|torque| max (N m)",
            "steps above 95 % of limit", "|joint vel| p99 (rad/s)", "|joint vel| max (rad/s)"], rows)
for name in ("torque_ratio_p99", "joint_speed_p99", "action_rate", "joint_acceleration", "mean_power"):
    print(f"{name}: {fmt(seed_means(baseline_metrics, name))} {METRICS[name][1]}")

**[ANSWER]** *Write your answer to the questions here, 2 paragraphs max*

### 2.1c Discussion: deployability

Given your previous observations, would your policy work on the real Go2? 
Discuss:
- **Observations:** can each observation be obtained on the real robot, and how noisy would it be?
- **Actions:** are the commanded motions realistic for the motors (look at torques, joint velocities and action smoothness)?
- **Evaluation on hardware:** which of your metrics could you measure on the real robot, and how?
- **Sim-to-real:** which reward terms or settings did you add to make the policy more transferable, and what is still missing?


**[ANSWER]** *Write your discussion here, 3 paragraphs max*

## 2.2 Controlled experiment

In this section you will describe and report the results of your controlled experiment in section 1.5 of notebook 01 like a short scientific study.

### 2.2a Questions and hypotheses

State your research question and your hypotheses:
- **Null hypothesis $H_0$:** the change has no effect on the metrics you chose.
- **Alternative hypothesis $H_1$:** the change affects the metrics, in a direction you predict, and why.

Name the metrics that will enable to reject the null hypothesis in favor of your alternative hypothesis, answering your research question.

**[ANSWER]** *Write your question, hypotheses, and metrics here.*

### 2.2b Methods

Describe exactly what differs between your baseline and your variant, what was held fixed (training budget, evaluation commands, seeds), how many runs each configuration has, and how you decide whether a difference is real (e.g. larger than the spread between seeds). 
Mention possible confounders.

**[ANSWER]** *Write your methods here. 2 paragraphs max.*

### 2.2c Experimental results

**Minimum:** at least 3 figures comparing baseline and variant, using training curves and/or evaluation metrics. 
Show both configurations in the same figure where possible, with the same axes.

In [ ]:
VARIANT = "no_base_lin_vel"
variant = load_config(VARIANT)

for seed, d in variant.items():
    r = d["all"]
    print(f"seed {seed}: run {d['run'].path.name}, survival {np.mean(r.alive):.1%} ({r.alive.sum()}/{r.num_robots})")
    # Same evaluation as the baseline (commands, robots per command, duration, evaluation seed)
    assert {k: v for k, v in r.meta.items() if k != "checkpoint"} == EVAL_META, seed

variant_metrics = {s: robot_metrics(d["alive"]) for s, d in variant.items()}
variant_survival = np.array([np.mean(variant[s]["all"].alive) for s in SEEDS])
CONFIGS = {"baseline": (baseline, baseline_metrics), VARIANT: (variant, variant_metrics)}

In [ ]:
# Baseline vs variant: every metric, per-seed means, and whether the difference exceeds the seed spread
def compare_row(name, unit, b, v, digits=3):
    diff = v.mean() - b.mean()
    spread = max(np.ptp(b), np.ptp(v))  # largest range between seeds of one configuration
    separated = b.max() < v.min() or v.max() < b.min()  # seed ranges do not overlap
    return [name, unit, fmt(b, digits), fmt(v, digits), f"{diff:+.3g} ({100 * diff / abs(b.mean()):+.0f} %)",
            f"{abs(diff) / spread:.1f}" if spread > 0 else ("0" if diff == 0 else "inf"), "yes" if separated else "no"]


rows = [compare_row("survival rate", "%", 100 * baseline_survival, 100 * variant_survival)]
for tag, name in [("Train/mean_reward", "train: final mean reward"),
                  ("Metrics/.*error_vel_xy", "train: final error_vel_xy"),
                  ("Metrics/.*error_vel_yaw", "train: final error_vel_yaw")]:
    t = find_tag(baseline, tag)
    rows.append(compare_row(name, "", final_values(baseline, t), final_values(variant, t)))
for name, (_, unit) in METRICS.items():
    rows.append(compare_row(name, unit, seed_means(baseline_metrics, name), seed_means(variant_metrics, name)))
show_table(["metric", "unit", "baseline", VARIANT, "difference (variant - baseline)",
            "|difference| / seed range", "seed ranges disjoint"], rows)
print("Values: mean ± sample std over seeds 42, 1, 2 of the per-seed mean. "
      "A difference is considered real when the seed ranges are disjoint (|difference| / seed range > 1 is a rough effect size).")

In [ ]:
# Figure 7: tracking error per command, baseline vs variant
fig, axes = plt.subplots(2, 1, figsize=(15, 7.5), sharex=True)
x = np.arange(len(COMMANDS))
for ax, (name, unit) in zip(axes, [("lin_vel_error", "m/s"), ("yaw_rate_error", "rad/s")]):
    for k, (cfg, (data, mets)) in enumerate(CONFIGS.items()):
        v = command_matrix(mets, data, name)
        ax.bar(x + (k - 0.5) * 0.4, v.mean(0), yerr=v.std(0, ddof=1), width=0.38,
               color=COLORS[cfg], capsize=2, label=f"{cfg} (mean ± std over seeds)")
    ax.set_ylabel(f"RMS {name} ({unit})")
    ax.set_title(f"{name} per command")
    ax.set_ylim(top=1.25 * ax.get_ylim()[1])  # room for the legend
    ax.grid(axis="y", alpha=0.3)
    ax.legend(fontsize=8, loc="upper left", ncol=2)
axes[1].set_xticks(x, [command_label(c) for c in COMMANDS], rotation=60, fontsize=8)
axes[1].set_xlabel("command (vx m/s, vy m/s, wz rad/s)")
fig.suptitle("Tracking error per command: baseline vs no_base_lin_vel")
fig.tight_layout()
plt.show()

# Per category, and the commands where the variant loses the most
rows = []
for name in ("lin_vel_error", "yaw_rate_error"):
    b = category_matrix(baseline_metrics, baseline, name)
    v = category_matrix(variant_metrics, variant, name)
    for k, cat in enumerate(CATEGORIES):
        rows.append([name, cat, fmt(b[:, k]), fmt(v[:, k]), f"{v[:, k].mean() - b[:, k].mean():+.3f}"])
show_table(["metric", "command category", "baseline", VARIANT, "difference"], rows)

d_lin = command_matrix(variant_metrics, variant, "lin_vel_error").mean(0) - lin_cmd.mean(0)
print("Largest increase of the linear velocity error (variant - baseline, m/s):")
for k in np.argsort(-d_lin)[:5]:
    print(f"   {command_label(COMMANDS[k]):>16} [{command_category(COMMANDS[k])}]: {d_lin[k]:+.3f}")

In [ ]:
# Figure 8: velocity error over time since the start of the rollout (all robots start standing)
def error_over_time(data):
    """(seeds, T) mean over moving-command robots of the linear velocity error at each step."""
    return np.stack([metrics.lin_vel_error_per_step(data[s]["alive"])[moving_mask(data[s]["alive"])].mean(0)
                     for s in SEEDS])


def mean_velocity(data, command, axis):
    """(seeds, T) mean body-frame velocity along `axis` of the robots with `command`."""
    out = []
    for s in SEEDS:
        r = data[s]["alive"]
        mask = np.all(np.isclose(r.cmd, command), axis=1)
        out.append(r.base_lin_vel[mask, :, axis].mean(0))
    return np.stack(out)


r0 = baseline[SEEDS[0]]["alive"]
t = r0.dt * np.arange(r0.base_lin_vel.shape[1])
FWD = (1.0, 0.0, 0.0)
ZOOM = 2.0  # s

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for cfg, (data, _) in CONFIGS.items():
    e = error_over_time(data)
    for ax in axes[:2]:
        ax.plot(t, e.mean(0), color=COLORS[cfg], lw=1.5, label=f"{cfg} (mean ± std over seeds)")
        ax.fill_between(t, e.mean(0) - e.std(0, ddof=1), e.mean(0) + e.std(0, ddof=1),
                        color=COLORS[cfg], alpha=0.25, lw=0)
    vx, vy = mean_velocity(data, FWD, 0), mean_velocity(data, FWD, 1)
    axes[2].plot(t, vx.mean(0), color=COLORS[cfg], lw=1.5, label=f"{cfg}: vx")
    axes[2].plot(t, vy.mean(0), color=COLORS[cfg], lw=1.5, ls="--", label=f"{cfg}: vy")
axes[0].set(title="Linear velocity error, all translating commands", xlim=(0, t[-1]))
axes[1].set(title=f"Zoom on the first {ZOOM:g} s", xlim=(0, ZOOM))
for ax in axes[:2]:
    ax.set(xlabel="time since rollout start (s)", ylabel="|v_xy - v_cmd| (m/s)")
axes[2].axhline(FWD[0], color="gray", ls=":", lw=1, label="commanded vx")
axes[2].axhline(0, color="gray", lw=0.8)
axes[2].set(xlim=(0, ZOOM), xlabel="time since rollout start (s)", ylabel="body velocity (m/s)",
            title=f"Mean velocity, command {command_label(FWD)}")
for ax in axes:
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
fig.suptitle("Start-up transient from standstill: baseline vs no_base_lin_vel")
fig.tight_layout()
plt.show()

# Transient numbers, per seed then mean ± std over seeds
early, late = t < 1.0, t >= 5.0
rows = []
for cfg, (data, _) in CONFIGS.items():
    e = error_over_time(data)
    steady = e[:, late].mean(1)
    settle = []
    for s in range(len(SEEDS)):
        smooth = np.convolve(e[s], np.ones(10) / 10, mode="same")
        above = np.flatnonzero(smooth > 1.2 * steady[s])
        if not len(above):
            settle.append(0.0)  # already within 1.2 x steady state from the start
        elif above[-1] + 1 < len(t):
            settle.append(t[above[-1] + 1])
        else:
            settle.append(np.nan)  # never settles
    vx = mean_velocity(data, FWD, 0)
    vy = mean_velocity(data, FWD, 1)
    rows.append([cfg, fmt(e[:, early].mean(1)), fmt(steady), fmt(settle),
                 fmt(vx[:, t < 1.0].min(1)), fmt(np.abs(vy[:, t < 1.0]).max(1))])
show_table(["configuration", "error 0 to 1 s (m/s)", "error 5 to 10 s (m/s)",
            "settling time (s, error within 1.2 x steady state)",
            f"min vx in first 1 s, cmd {command_label(FWD)} (m/s)",
            f"max |vy| in first 1 s, cmd {command_label(FWD)} (m/s)"], rows)

In [ ]:
# Figure 9: training curves, baseline vs variant
PANELS_TRAIN = [("Train/mean_reward", "Mean episode reward (-)"),
                ("Episode_Reward/track_linear_velocity", "Reward per episode (-)"),
                ("Metrics/.*error_vel_xy", "error_vel_xy (m/s)"),
                ("Metrics/.*error_vel_yaw", "error_vel_yaw (rad/s)")]
fig, axes = plt.subplots(1, 4, figsize=(17, 3.8))
for ax, (regex, ylabel) in zip(axes, PANELS_TRAIN):
    tag = find_tag(baseline, regex)
    for cfg, (data, _) in CONFIGS.items():
        plot_band(ax, data, tag, label=f"{cfg} (mean ± std)", color=COLORS[cfg])
    ax.set(xlabel="Learning iteration", ylabel=ylabel, title=tag.split("/")[-1])
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
fig.suptitle("Training curves: baseline vs no_base_lin_vel (3 seeds each)")
fig.tight_layout()
plt.show()

# Iteration at which the training velocity error first gets within 10 % of its final value
tag = find_tag(baseline, "Metrics/.*error_vel_xy")
for cfg, (data, _) in CONFIGS.items():
    it, v = curves(data, tag)
    p = np.array([progress_iteration(it, s, 0.9) for s in v])
    print(f"{cfg}: error_vel_xy covers 90 % of its decrease at iteration {fmt(p)}")

In [ ]:
# Figure 10: gait, efficiency and stability metrics (one dot per seed, bar = mean over seeds)
PANELS_GAIT = ["duty_factor", "stride_frequency", "foot_clearance",
               "cost_of_transport", "base_ang_vel_xy", "action_rate"]
fig, axes = plt.subplots(2, 3, figsize=(12, 6.5))
for ax, name in zip(axes.flat, PANELS_GAIT):
    for k, (cfg, (_, mets)) in enumerate(CONFIGS.items()):
        v = seed_means(mets, name)
        ax.bar(k, v.mean(), width=0.6, color=COLORS[cfg], alpha=0.6, label=cfg)
        ax.plot(np.full(len(v), k), v, "o", color="black", ms=4, label="single seed" if k == 0 else None)
    ax.set_xticks([0, 1], list(CONFIGS))
    ax.set(ylabel=f"{name} ({METRICS[name][1]})", title=name)
    ax.grid(axis="y", alpha=0.3)
axes.flat[0].legend(fontsize=8)
fig.suptitle("Gait and stability over the command grid (robots that never fell)")
fig.tight_layout()
plt.show()

variant_sync = pair_sync(variant)
show_table(["foot pair", "baseline", VARIANT],
           [[k, fmt(baseline_sync[k]), fmt(variant_sync[k])] for k in PAIRS])

**[ANSWER]** *Describe what each figure shows here. 2 paragraphs max.*

### 2.2e Discussion

What did you expect, and what did you not expect? 
Do the results support $H_1$ and reject $H_0$, and how conclusive are they (number of runs, size of the difference compared to the variation between runs)? 
Explain the mechanism: how does your change lead to the effect you measured? 
What would you test next?

**[ANSWER]** *Describe your interpretations here. Linking your observations back to the course or the literature is encouraged! 3 paragraphs max.*